# Indexing embeddings

## 1. Import libs

In [ ]:
import pickle
from pathlib import Path
import torch
import os
from dotenv import load_dotenv
from elasticsearch import Elasticsearch

## 2. Combination: Train, validation, test dataset

In [ ]:
test_embedding_path = Path("../data/repo_embeddings/repo_info_test_embeddings_reduce.pkl")
with test_embedding_path.open("rb") as f:
    test_info = pickle.load(f)

validation_embedding_path = Path("../data/repo_embeddings/repo_info_validation_embeddings_reduce.pkl")
with validation_embedding_path.open("rb") as f:
    validation_info = pickle.load(f)
    
train_embedding_path = Path("../data/repo_embeddings/repo_info_train_embeddings_reduce.pkl")
with train_embedding_path.open("rb") as f:
    train_info = pickle.load(f)

def ensure_2d(t):
    t = torch.tensor(t)
    if t.ndim == 1:
        t = t.unsqueeze(0)
    return t

repo_info = dict()
for embedding in [test_info, validation_info, train_info]:
    for repo, info in embedding.items():
        repo_info[repo] = dict()
        
        codes_embeddings = ensure_2d(info["codes_embeddings"])
        mean_code_embedding = torch.mean(codes_embeddings, dim=0, keepdim=True)
        repo_info[repo]["codes_embeddings"] = codes_embeddings
        repo_info[repo]["mean_code_embedding"] = mean_code_embedding
        
        docs_embeddings = ensure_2d(info["docs_embeddings"])
        mean_doc_embedding = torch.mean(docs_embeddings, dim=0, keepdim=True)
        repo_info[repo]["docs_embeddings"] = docs_embeddings
        repo_info[repo]["mean_doc_embedding"] = mean_doc_embedding
        
        requirements_embeddings = ensure_2d(info["requirements_embeddings"])
        mean_requirements_embedding = torch.mean(requirements_embeddings, dim=0, keepdim=True)
        repo_info[repo]["requirements_embeddings"] = requirements_embeddings
        repo_info[repo]["mean_requirements_embedding"] = mean_requirements_embedding
        
        readme_embeddings = ensure_2d(info["readme_embeddings"])
        mean_readme_embedding = torch.mean(readme_embeddings, dim=0, keepdim=True)
        repo_info[repo]["readme_embeddings"] = readme_embeddings
        repo_info[repo]["mean_readme_embedding"] = mean_readme_embedding
        
        repo_info[repo]["mean_repo_embeddings"] = torch.concatenate([
            mean_code_embedding,
            mean_doc_embedding,
            mean_requirements_embedding,
            mean_readme_embedding
        ], dim=0).reshape(1, -1)

with Path("../data/repo_embeddings/repo_info_embeddings.pkl").open("wb") as f:
    pickle.dump(repo_info, f)

In [ ]:
with Path("../data/repo_embeddings/repo_info_embeddings.pkl").open("rb") as f:
    repo_info_embeddings = pickle.load(f)
    
len(repo_info_embeddings.items())

## 3. Indexing embeddings into Elastic Search

In [ ]:
load_dotenv()
INDEX_PROCESSED = "repositories_processed"
ES_URL = os.getenv("ES_URL", "http://localhost:9200")
API_KEY = os.getenv("ES_API_KEY")

In [ ]:
def get_client() -> Elasticsearch:
    return Elasticsearch(ES_URL, api_key=API_KEY)

In [ ]:
import time
from elasticsearch import NotFoundError

es = get_client()
count = 0
excluded_repos = []

for repo_id, repo_info in repo_info_embeddings.items():
    if not es.exists(index=INDEX_PROCESSED, id=repo_id):
        excluded_repos.append(repo_id)
        continue

    vec = repo_info["mean_repo_embeddings"].squeeze(0).cpu().numpy().astype(float).tolist()
    try:
        resp = es.update(
            index=INDEX_PROCESSED,
            id=repo_id,
            doc={"embedding": vec},
            request_timeout=60,
            refresh="wait_for"
        )
        print("updated:", repo_id, resp.get("result"))
        count += 1
    except NotFoundError:
        excluded_repos.append(repo_id)
        continue
    except Exception as e:
        print(f"Failed update {repo_id}: {e}")
        time.sleep(1)

    time.sleep(0.05)

print("updated:", count)

In [ ]:
with Path("../data/excluded_repos.txt").open("wb") as f:
    for repo_id in excluded_repos:
        f.write(f"{repo_id}\n".encode())

In [ ]:
# validation
doc = es.get(index=INDEX_PROCESSED, id="0rpc/zerorpc-python")
print("embedding present:", "embedding" in doc["_source"], "len:", len(doc["_source"].get("embedding", [])))